In [ ]:
import pandas as pd
import numpy as np
import torch
from torch import nn, optim
from torch.utils.data import DataLoader

X_test = np.load('../data/processed/X_test.npy')
y_test = np.load('../data/processed/y_test.npy')
X_val = np.load('../data/processed/X_val.npy')
y_val = np.load('../data/processed/y_val.npy')
X_train_normal = np.load('../data/processed/X_train_normal.npy')
y_train = np.load('../data/processed/y_train.npy')


# Entrenamos el Autoencoder solo con casos normales (y == 0)
train_tensor = torch.FloatTensor(X_train_normal)

# DataLoader para entrenamiento eficiente
train_loader = DataLoader(train_tensor, batch_size=256, shuffle=True)


In [3]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super(Autoencoder, self).__init__()
        # Encoder: 29 -> 16 -> 8 -> 4
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 4),
            nn.ReLU()
        )

        # Decoder: 4 -> 8 -> 16 -> 29
        self.decoder = nn.Sequential(
            nn.Linear(4, 8),
            nn.ReLU(),
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, input_dim) # Sin ReLU final
        )
    
    def forward(self, x):
        return self.decoder(self.encoder(x))



In [4]:
model = Autoencoder(input_dim=30)
epochs = 50
criterion = nn.MSELoss()  # Mean Squared Error
optimizer = optim.Adam(model.parameters(), lr=1e-3)

model.train() 
for epoch in range(epochs):  
    train_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad() # Resetrar el gradiente acumulado a 0
        output = model(batch) # Forward Pass
        loss = criterion(output, batch) # Calculamos la perdida
        loss.backward() # Calculamos los gradientes
        optimizer.step() # Actualizamos los pesos
        train_loss += loss.item() # Acumulamos la perdida del batch
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {train_loss/len(train_loader):.4f}')

Epoch [10/50], Loss: 0.6209
Epoch [20/50], Loss: 0.5886
Epoch [30/50], Loss: 0.5785
Epoch [40/50], Loss: 0.5694
Epoch [50/50], Loss: 0.5638


In [5]:
model.eval()
with torch.no_grad():
    X_test_tensor = torch.FloatTensor(X_test)
    recon = model(X_test_tensor)
    recon_error = ((recon - X_test_tensor) ** 2).mean(dim=1).numpy()


In [35]:
from sklearn.metrics import confusion_matrix

y_pred = (recon_error >= 1.744).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()


print("TP (fraude detectado):", tp)
print("FN (fraude perdido):", fn)
print("FP (normal marcado como fraude):", fp)
print("TN (normal correcto):", tn)


TP (fraude detectado): 42
FN (fraude perdido): 7
FP (normal marcado como fraude): 815
TN (normal correcto): 27617


In [33]:
precision = tp / (tp + fp)
recall = tp / (tp + fn)
print("Precision:", precision)
print("Recall:", recall)

Precision: 0.049008168028004666
Recall: 0.8571428571428571


In [2]:
porcentaje_anomailias = tp / (tp + fn)
porcentaje_normales = tn /(tn + fp)
print(porcentaje_anomailias, porcentaje_normales)

NameError: name 'tp' is not defined